# LSTM — one config, one run folder, one scored result

Trains the config named in **the parameter cell below** into an immutable run
folder under `src/model/runs/`, scores it with `result_evaluator` and draws the
four diagnostic figures.

The notebook holds **no logic `train.py` does not** — it calls the same
`train()`, so a sweep is a shell loop and not nine edited copies of this file:

```
python -m model.lstm --config configs/lstm__vcb__close_adjust_5day__final__d20_h5.yaml
```

> ⚠️ `lookback` and `n_features` in the config are ASSERTIONS. `train()` raises
> if the dataset disagrees — the dataset is the authority.

> ⚠️ **The default target here, `close_adjust_5day`, is a price LEVEL and its
> selection FAILED its own null** (2026-08-16: IC −0.0608 against a +0.0853 bar,
> p = 0.7273). Every label is positive, so `hit_rate` and `dir_accuracy` are
> 1.0 by construction and unreadable; `ic` against its null is the only metric
> worth quoting. This chain is exercised end to end here — it is not evidence.


## Import

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, os.path.abspath("../.."))   # src/ -> model, result_evaluator

from model.lstm.train import CONFIG_DIR, load_config, train
from model.common.data import load_dataset
from result_evaluator import metrics as M
from result_evaluator.evaluator import evaluate_run, leaderboard
from result_evaluator.plots import run_figures, use_theme

use_theme()
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

## Config — the parameter cell

⚠️ **Retarget by editing the four names below, never by editing a path string.**
The config file, the dataset folder and the run name all derive from
`(TICKER, TARGET, D, H)`, so they cannot drift apart. `CONFIG_PATH` in the
environment still wins, which is how `python -m model.lstm --config …` and this
notebook stay the same run.

One YAML fully describes the run. Its `dataset` field names a folder under
`src/train_test_set/` built by
`python -m train_test_creator --table <TARGET>__final__d<D>_h<H> --save`.


In [ ]:
# ----------------------------------------------------------------- parameters
TICKER = "vcb"                    # unified_schema_<TICKER>
TARGET = "close_adjust_5day"      # the label column in pool__targets
D, H = 20, 5                      # lookback, horizon — read from the TABLE NAME
MODEL = "lstm"                    # the config-name prefix, = the model package
# -----------------------------------------------------------------------------

TABLE = f"{TARGET}__final__d{D}_h{H}"
CONFIG_NAME = f"{MODEL}__{TICKER}__{TABLE}.yaml"

# ⚠️ The `{MODEL}__` prefix is load-bearing and was missing here until 2026-08-16,
# which made this cell's default point at a file that does not exist — the
# notebook could only run with CONFIG_PATH already set. CLAUDE.md §5 rule 10 at
# the filesystem: assert the path resolves rather than trusting that it does.
CONFIG_PATH = os.environ.get("CONFIG_PATH", os.path.join(CONFIG_DIR, CONFIG_NAME))
assert os.path.exists(CONFIG_PATH), (
    f"no config at {CONFIG_PATH}. Configs present: "
    f"{sorted(f for f in os.listdir(CONFIG_DIR) if f.endswith('.yaml'))}"
)

CONFIG = load_config(CONFIG_PATH)
CONFIG


## The dataset, and where it came from

`lineage` is the chain this run is allowed to claim: source table → the table's
`COMMENT` → dataset → run. It travels into `metadata.json`, so a run folder read
months from now still says what its features are **and are not**.

In [ ]:
dataset = load_dataset(CONFIG["dataset"])
source = dataset.meta["source"]

print(f"dataset  {dataset.name}")
print(f"hash     {dataset.hash}")
print(f"source   {source['schema']}.{source['table']}")
print(f"window   d={dataset.lookback}  features={dataset.n_features}")
print(f"samples  train {len(dataset.y_train)} | val {len(dataset.y_val)}"
      f" | test {len(dataset.y_test)}")
print()
print("evidence:", dataset.meta["evidence"])

## Train

Creates the run folder, trains with early stopping on val loss, writes
`predictions_{val,test}.csv`, scores them and appends a row to
`runs/index.csv`. Everything below reads that folder back.

In [ ]:
run_dir, metrics = train(CONFIG)
run_dir

## The scored result

The same four **core** metrics every model type reports — `ic`, `dir_auc`,
`dir_accuracy`, `long_short` — each against a block-shuffled null, plus the
regression extras. See `result_evaluator/CONTEXT.md` §2 for why these four.

In [ ]:
metrics

In [ ]:
for split in metrics.index:
    print(f"{split}: {M.verdict(metrics.loc[split].to_dict())}\n")

## ⚠️ The bar, drawn

`ic` against 200 block-shuffled reruns. The block is `d + h = 25` rows — one
sample's whole footprint — because a row-wise shuffle would destroy the label's
own autocorrelation and produce a bar far too low to fail.

In [ ]:
predictions = pd.read_csv(os.path.join(run_dir, "results", "predictions_test.csv"))
null = M.null_draws(
    predictions["y_true"].to_numpy(),
    predictions["y_pred"].to_numpy(),
    block=dataset.lookback + dataset.meta["target"]["horizon_h"],
    metric="ic",
    draws=200,
)
print(f"observed {null['observed']:+.4f}  bar {null['bar']:+.4f}"
      f"  p {null['p']:.3f}  ({len(null['draws'])} usable draws)")

## The four figures

| | answers |
|---|---|
| top-left | did it learn, or stop at epoch 1? |
| top-right | is the relationship in the tails, where it would be traded? |
| bottom-left | what does the prediction series look like over time? |
| bottom-right | is `ic` outside what shuffled labels produce? |

Saved to `results/figures_test.png` in the run folder.

In [ ]:
figure = run_figures(run_dir, "test", draws=null)
plt.show()

## Every run, on one board

Sorted by `ic` — which is **not** an endorsement. `ic_clears` beside it is what
says whether the ordering means anything.

In [ ]:
board = leaderboard()
columns = [
    "run_id", "model", "task", "lookback", "n_eff",
    "ic", "ic_p", "ic_clears", "dir_auc", "dir_auc_p", "dir_auc_clears",
    "long_short", "return_source",
]
board[[c for c in columns if c in board.columns]]

## ⚠️ What a cleared bar does and does not mean

The null here shuffles the **outcome** against a fixed score vector. By the time
it runs, the features were selected, the architecture was chosen and the epoch
was early-stopped on val — none of which it prices in. Unlike
`feature_selection.evaluation.null_distribution`, it cannot re-run the selection.
That is issue `NUL-1`.

**A run that fails this bar is dead. A run that clears it is not yet alive.**

And the features carry a warning one stage further up. For the default target
`close_adjust_5day` (as rebuilt 2026-08-16):

| | |
|---|---|
| source runs | **1** layer-2 run over `pool__shortlist__close_adjust_5day__d20_h5` |
| channels | **35**, competed rather than unioned |
| evidence | **`failed_null=1`** — a measurement, not an unknown |
| the selection's own bar | IC −0.0608 vs p95 +0.0853, null mean −0.0123, z −0.61, p 0.7273 |
| the top channel | `russia__economy__money__economics__rulps`, ρ 0.967 to the target — it absorbed `close_adjust` itself as redundant |
| drift (`DRF-1`) | 8 of 35 channels put >1% of TEST beyond 5 train-sigmas; 1 puts **all** of it there |

⚠️ Read `dataset.meta["evidence"]` printed above rather than this table — it is
generated from the table's own `COMMENT` and cannot go stale the way prose does.
